In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================
# CONFIGURACIÓN
# ==========================================

TABLA_SILVER = "quality_data.silver.control_calidad"

TABLA_LIMITES = "quality_data.gold.limites_control"
TABLA_EVALUADA = "quality_data.gold.control_calidad_evaluado"
TABLA_ALERTAS = "quality_data.gold.alertas_calidad"
TABLA_RESUMEN = "quality_data.gold.resumen_alertas_calidad"

df_silver = spark.table(TABLA_SILVER)

print("=" * 60)
print("CARGA DE SILVER")
print("=" * 60)
print(f"Tabla origen: {TABLA_SILVER}")
print(f"Registros Silver: {df_silver.count():,}")
print(f"Columnas: {len(df_silver.columns)}")

display(df_silver.limit(20))

In [0]:
# ==========================================
# CREAR ESQUEMA GOLD
# ==========================================

spark.sql("CREATE CATALOG IF NOT EXISTS quality_data")
spark.sql("CREATE SCHEMA IF NOT EXISTS quality_data.gold")

print("✅ Esquema quality_data.gold disponible")

In [0]:
# ==========================================
# DATOS BASE PARA CÁLCULO DE LÍMITES
# ==========================================

df_base_limites = (
    df_silver
    .filter(
        F.col("articulo_codigo").isNotNull()
        & (F.trim(F.col("articulo_codigo")) != "")
        & F.col("variable_codigo").isNotNull()
        & (F.trim(F.col("variable_codigo")) != "")
        & F.col("resultado_numerico").isNotNull()
        & F.col("fecha_medicion").isNotNull()
    )
)

print("=" * 60)
print("BASE PARA LÍMITES")
print("=" * 60)
print(f"Registros disponibles: {df_base_limites.count():,}")

display(df_base_limites.limit(20))

In [0]:
# ==========================================
# CALCULAR LÍMITES POR ARTÍCULO Y VARIABLE_NOMBRE
# ==========================================

MIN_MEDICIONES = 5

df_limites_control = (
    df_base_limites
    .groupBy(
        "articulo_codigo",
        "variable_nombre"
    )
    .agg(
        F.count("*").alias("total_mediciones"),
        F.countDistinct("hdr").alias("total_hdr"),
        F.round(F.avg("resultado_numerico"), 6).alias("promedio"),
        F.round(F.stddev("resultado_numerico"), 6).alias("desviacion_estandar"),
        F.min("resultado_numerico").alias("valor_minimo_historico"),
        F.max("resultado_numerico").alias("valor_maximo_historico"),
        F.min("fecha_medicion").alias("fecha_minima"),
        F.max("fecha_medicion").alias("fecha_maxima")
    )
    .withColumn(
        "limite_inferior",
        F.round(
            F.col("promedio") - (F.lit(3) * F.col("desviacion_estandar")),
            6
        )
    )
    .withColumn(
        "limite_superior",
        F.round(
            F.col("promedio") + (F.lit(3) * F.col("desviacion_estandar")),
            6
        )
    )
    .withColumn(
        "suficientes_datos",
        F.when(
            F.col("total_mediciones") >= MIN_MEDICIONES,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "metodo_calculo",
        F.lit("PROMEDIO +/- 3 DESVIACIONES")
    )
    .withColumn(
        "fecha_calculo_limites",
        F.current_timestamp()
    )
)

display(
    df_limites_control
    .orderBy(F.desc("total_mediciones"))
)

In [0]:
# ==========================================
# GUARDAR LÍMITES DE CONTROL
# ==========================================

spark.sql(f"DROP TABLE IF EXISTS {TABLA_LIMITES}")

df_limites_control.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_LIMITES)

print(f"✅ Tabla creada: {TABLA_LIMITES}")

In [0]:
# ==========================================
# EVALUAR MEDICIONES CONTRA LÍMITES
# ==========================================

df_evaluado = (
    df_base_limites.alias("s")
    .join(
        df_limites_control.alias("l"),
        on=[
            "articulo_codigo",
            "variable_nombre"
        ],
        how="left"
    )
    .withColumn(
        "estado_control",
        F.when(
            F.col("suficientes_datos") == False,
            F.lit("SIN_DATOS_SUFFICIENTES")
        )
        .when(
            F.col("desviacion_estandar").isNull(),
            F.lit("SIN_DESVIACION")
        )
        .when(
            F.col("resultado_numerico") < F.col("limite_inferior"),
            F.lit("BAJO_LIMITE")
        )
        .when(
            F.col("resultado_numerico") > F.col("limite_superior"),
            F.lit("SUPERA_LIMITE")
        )
        .otherwise(
            F.lit("DENTRO_LIMITE")
        )
    )
    .withColumn(
        "es_fuera_limite",
        F.when(
            F.col("estado_control").isin("BAJO_LIMITE", "SUPERA_LIMITE"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "distancia_al_limite",
        F.when(
            F.col("estado_control") == "BAJO_LIMITE",
            F.round(F.col("limite_inferior") - F.col("resultado_numerico"), 6)
        )
        .when(
            F.col("estado_control") == "SUPERA_LIMITE",
            F.round(F.col("resultado_numerico") - F.col("limite_superior"), 6)
        )
        .otherwise(F.lit(0))
    )
    .withColumn(
        "fecha_evaluacion",
        F.current_timestamp()
    )
)

display(
    df_evaluado.select(
        "hdr",
        "articulo_codigo",
        "variable_nombre",
        "resultado_numerico",
        "limite_inferior",
        "limite_superior",
        "estado_control",
        "es_fuera_limite",
        "fecha_medicion"
    ).limit(1000)
)

In [0]:
# ==========================================
# GUARDAR TABLA EVALUADA
# ==========================================

spark.sql(f"DROP TABLE IF EXISTS {TABLA_EVALUADA}")

df_evaluado.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_EVALUADA)

print(f"✅ Tabla creada: {TABLA_EVALUADA}")

In [0]:
# ==========================================
# CREAR TABLA DE ALERTAS
# ==========================================

df_alertas = (
    df_evaluado
    .filter(F.col("es_fuera_limite") == True)
    .select(
        "hdr",
        "cliente_codigo",
        "articulo_codigo",
        "variable_nombre",
        "resultado_original",
        "resultado_numerico",
        "limite_inferior",
        "limite_superior",
        "estado_control",
        "distancia_al_limite",
        "color_codigo",
        "color_nombre",
        "fecha_medicion",
        "fecha_medicion_dia",
        "anio_medicion",
        "mes_medicion",
        "semana_medicion",
        "total_mediciones",
        "promedio",
        "desviacion_estandar",
        "metodo_calculo",
        "fecha_evaluacion"
    )
)

spark.sql(f"DROP TABLE IF EXISTS {TABLA_ALERTAS}")

df_alertas.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_ALERTAS)

print(f"✅ Tabla creada: {TABLA_ALERTAS}")
print(f"Alertas detectadas: {df_alertas.count():,}")

display(df_alertas.limit(4000))

In [0]:
# ==========================================
# RESUMEN PARA DASHBOARD
# ==========================================

df_resumen_alertas = (
    df_evaluado
    .groupBy(
        "articulo_codigo",
        "variable_nombre"
    )
    .agg(
        F.count("*").alias("total_mediciones"),
        F.sum(
            F.when(F.col("es_fuera_limite") == True, 1).otherwise(0)
        ).alias("total_fuera_limite"),
        F.round(
            F.sum(
                F.when(F.col("es_fuera_limite") == True, 1).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias("porcentaje_fuera_limite"),
        F.round(F.avg("resultado_numerico"), 6).alias("promedio_resultado"),
        F.min("resultado_numerico").alias("valor_minimo"),
        F.max("resultado_numerico").alias("valor_maximo"),
        F.min("fecha_medicion").alias("fecha_minima"),
        F.max("fecha_medicion").alias("fecha_maxima")
    )
    .orderBy(F.desc("total_fuera_limite"))
)

spark.sql(f"DROP TABLE IF EXISTS {TABLA_RESUMEN}")

df_resumen_alertas.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_RESUMEN)

print(f"✅ Tabla creada: {TABLA_RESUMEN}")

display(df_resumen_alertas)